# One-qubit playground — a phase is invisible until it interferes

**The punchline.** A $Z$ gate does *nothing you could ever detect* to $\lvert 0\rangle$,
and *everything* to $\lvert +\rangle$ — and even on $\lvert +\rangle$ its effect stays
hidden until a second Hadamard folds the two halves of the superposition back together.
Phases are not observable. Phase *differences*, run through an interferometer, are the
only thing that is.

This exhibit is one qubit, six gates, and three plots. Background: **[01 — States and
gates](../01-states-and-gates.ipynb)** introduces the qubit, Dirac notation and the Bloch
sphere from scratch; this notebook assumes that and nothing more.

## The three numbers that describe a qubit

A single qubit's state is two complex numbers, $a\lvert 0\rangle + b\lvert 1\rangle$,
with $|a|^2 + |b|^2 = 1$. That is four real numbers, minus one for the normalisation,
minus one more for an *overall* phase that no experiment can see — so three real numbers
are left. Those three are the **Bloch vector** $(x, y, z)$, and they are the average
values you would get by measuring the qubit along each of three perpendicular axes:

$$x = \langle X\rangle,\qquad y = \langle Y\rangle,\qquad z = \langle Z\rangle.$$

Geometrically: $\lvert 0\rangle$ sits at the north pole $(0,0,1)$, $\lvert 1\rangle$ at
the south pole, and $\lvert +\rangle = (\lvert 0\rangle + \lvert 1\rangle)/\sqrt2$ on the
equator at $(1,0,0)$. A pure state is always length 1 — on the surface of the ball.

`qc.inspect.bloch_vector(q)` hands them to you. It is a *cheat*: a real machine could
only estimate these three numbers by preparing the state thousands of times and measuring
along a different axis each time. Everything under `qc.inspect` is like that, which is
why it lives behind a name that says so.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, viz
from qsim.gates import H, Rx, Ry, Rz, X, Z


def one_qubit(*ops) -> Circuit:
    """A fresh one-qubit circuit with `ops` applied in order to its qubit.

    Fresh every time, on purpose. A gate is a physical operation, not a setting: "what
    does Z do to |+>" is a *different experiment* from "what does Z do to |0>", not an
    edit to the same one. Rebuilding is also what makes the sweeps further down honest.
    """
    qc = Circuit(name="playground", seed=0)
    q = qc.alloc("q")
    for op in ops:
        op(q)
    return qc

## A gallery, one gate at a time

Eight preparations, each starting from $\lvert 0\rangle$. For each one we print the Bloch
vector and $P(0)$ — the probability that a computational-basis measurement (the only kind
qsim's `measure` performs) reports 0.

Watch the last column, and then the first three.

In [ ]:
gallery = [
    ("|0>  (do nothing)", []),
    ("X|0>", [X]),
    ("H|0>", [H]),
    ("Z H|0>", [H, Z]),
    ("Rx(pi/2)|0>", [lambda q: Rx(q, theta=np.pi / 2)]),
    ("Ry(pi/2)|0>", [lambda q: Ry(q, theta=np.pi / 2)]),
    ("Rz(pi/2) H|0>", [H, lambda q: Rz(q, theta=np.pi / 2)]),
    ("Z|0>", [Z]),
]

print(f"{'preparation':>18}   {'x':>7} {'y':>7} {'z':>7}   {'P(0)':>6}")
for label, ops in gallery:
    qc = one_qubit(*ops)
    x, y, z = qc.inspect.bloch_vector(qc.qubits[0])
    p0 = qc.inspect.probabilities()[0]
    print(f"{label:>18}   {x:+7.3f} {y:+7.3f} {z:+7.3f}   {p0:6.3f}")

### Aside: two kinds of minus

The first two rows look paradoxical: their Bloch vectors are exact negatives of each
other, yet one is *certain* to read 0 and the other *certain* not to. If "minus the
state" is supposed to be invisible — and the end of this notebook makes exactly that
claim — how can these two rows differ so completely?

Because those are two different minuses, living in two different spaces.

As **amplitude pairs** the two states are not negatives at all: $\lvert 0\rangle = (1,0)$
and $X\lvert 0\rangle = \lvert 1\rangle = (0,1)$ are *orthogonal* — and antipodal points
on the sphere always are. The dictionary between overlap and Bloch geometry is

$$|\langle\varphi\vert\psi\rangle|^2 = \frac{1 + \vec v_\varphi\cdot\vec v_\psi}{2},$$

so opposite Bloch vectors ($\vec v_\varphi\cdot\vec v_\psi = -1$) mean overlap zero: the
**most distinguishable** pair of states there is, not the same state twice. The $z$
column even contains the last column: $P(0) = (1+z)/2$, so being antipodal along $z$
*forces* opposite certainties.

Negating the **amplitudes**, $\psi \to -\psi$, is the other minus — and it moves the
Bloch vector nowhere, because $x$, $y$, $z$ are each quadratic in the amplitudes, so a
global phase cancels out of all three. That invisible minus is where this notebook ends
($R_x(2\pi) = -I$).

Notice the factor of two running underneath: orthogonal states sit $90°$ apart in
amplitude space but $180°$ apart on the sphere. The map from amplitudes to the sphere
erases the global sign and **doubles every angle** — the same doubling as the half-angles
in every rotation gate's matrix, told in full in
[quaternions_and_spin](quaternions_and_spin.ipynb).

Two rows are worth staring at.

**`Z|0>` is identical to `|0>`.** Same Bloch vector, same probabilities, and in fact the
same state vector: $Z$ multiplies the $\lvert 1\rangle$ amplitude by $-1$, and
$\lvert 0\rangle$ has no $\lvert 1\rangle$ amplitude to multiply. Nothing happened.

**`Z H|0>` and `H|0>` have the same $P(0)$ — but opposite $x$.** These are
$\lvert +\rangle$ and $\lvert -\rangle$, the two opposite points on the equator. They are
as different as two states can be: a measurement along the $x$ axis distinguishes them
perfectly, every single time. But a measurement along $z$ — the only one a computational-
basis `measure()` performs — gives a fair coin for both. Same statistics, opposite states.

Here is that claim as an experiment. Both circuits carry the same seed, so they draw from
the same random stream; because their probabilities are also identical, the outcomes come
out *identical*, sample for sample.

In [ ]:
plus = one_qubit(H)        # |+>
minus = one_qubit(H, Z)    # |->

plus_counts = plus.inspect.sample(2000)
minus_counts = minus.inspect.sample(2000)

overlap = plus.inspect.overlap(minus.inspect.state_vector())

print("|+> sampled 2000 times:", dict(plus_counts))
print("|-> sampled 2000 times:", dict(minus_counts))
print()
print("the two outcome tallies are literally equal:", plus_counts == minus_counts)
print(f"and yet <-|+> = {abs(overlap):.3f}  (0 means perfectly distinguishable)")

`overlap` returns $\langle -\vert + \rangle = 0$: the two states are *orthogonal*, the
quantum version of "as different as possible". And a $z$-measurement cannot tell them
apart at all. That gap — between how different two states are and how different your
chosen measurement makes them look — is the whole subject of this notebook.

On the Bloch sphere the difference is impossible to miss.

In [ ]:
fig_plus = viz.bloch(plus, plus.qubits[0])
fig_minus = viz.bloch(minus, minus.qubits[0])

## Making the phase visible: the interferometer

To *see* the difference between $\lvert +\rangle$ and $\lvert -\rangle$ you have to
measure along $x$ instead of $z$ — and the way you do that with only computational-basis
measurement available is to rotate the state first. $H$ is exactly that rotation: it
swaps the $x$ and $z$ axes of the Bloch sphere, so $H\lvert +\rangle = \lvert 0\rangle$
and $H\lvert -\rangle = \lvert 1\rangle$.

Read as an experiment, the three-gate sequence

$$H \;\to\; R_z(\theta) \;\to\; H$$

is an interferometer. The first $H$ splits $\lvert 0\rangle$ into two paths. $R_z(\theta)$
puts a relative phase $\theta$ between them — invisible while they are apart. The second
$H$ recombines them, and the phase decides whether they add or cancel.

The sweep below builds a **fresh circuit for each of 121 angles** and records two things:
the Bloch vector just after the phase (before recombining), and $P(0)$ after recombining.

In [ ]:
thetas = np.linspace(0.0, 2.0 * np.pi, 121)
bloch_rows = []
p0_split = []       # P(0) measured while the two paths are still apart
p0_joined = []      # P(0) after the second H recombines them

for theta in thetas:
    qc = Circuit(name="interferometer", seed=0)
    q = qc.alloc("q")
    H(q)                       # split
    Rz(q, theta=theta)         # phase one path relative to the other
    bloch_rows.append(qc.inspect.bloch_vector(q))
    p0_split.append(qc.inspect.probabilities()[0])
    H(q)                       # recombine
    p0_joined.append(qc.inspect.probabilities()[0])

# A list of 121 three-tuples becomes a (121, 3) array; column k is one Bloch
# component as a function of theta.
bloch = np.array(bloch_rows)

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.0, 3.6))

ax_left.plot(thetas, bloch[:, 0], lw=2, label="x")
ax_left.plot(thetas, bloch[:, 1], lw=2, label="y")
ax_left.plot(thetas, bloch[:, 2], lw=2, label="z")
ax_left.plot(thetas, p0_split, "k--", lw=1.2, label="P(0), paths apart")
ax_left.set_title("after H then Rz(theta): the state moves,\nthe z-measurement does not")

ax_right.plot(thetas, p0_joined, lw=2, color="crimson", label="P(0), recombined")
ax_right.plot(thetas, np.cos(thetas / 2) ** 2, "k--", lw=1.2, label=r"$\cos^2(\theta/2)$")
ax_right.set_title("after the second H:\nthe phase has become a probability")

for ax in (ax_left, ax_right):
    ax.set_xlabel(r"$\theta$")
    ax.set_xticks([0, np.pi, 2 * np.pi], ["0", "π", "2π"])
    ax.set_ylim(-1.15, 1.35)
    ax.axhline(0.0, color="gray", lw=0.6)
    ax.legend(fontsize=8, ncol=2, loc="upper center")
fig.tight_layout()

**Left panel.** As $\theta$ runs from $0$ to $2\pi$ the Bloch vector walks all the way
round the equator: $x = \cos\theta$, $y = \sin\theta$, $z = 0$ throughout. The state is
changing continuously and dramatically. Meanwhile the dashed line — $P(0)$ for a
computational-basis measurement — is pinned at $0.5$ and never moves. A $z$-measurement
is blind to a walk around the equator, because $z$ is the one coordinate the walk leaves
alone.

**Right panel.** Recombine first, and the same walk becomes a full swing of the outcome
probability, $P(0) = \cos^2(\theta/2)$, from certainty at $\theta = 0$ to certainty of
the *other* answer at $\theta = \pi$. Nothing was added to the state between the two
panels; the only difference is whether the paths were brought back together before we
looked.

This is why "the phase is unobservable" and "the phase is everything" are both true, and
it is the mechanism every quantum algorithm runs on: arrange for the phases of the wrong
answers to cancel when the paths meet.

## The continuous sweep: $R_x(\theta)$ from $0$ to $4\pi$

The obvious way to *see* a rotation is to animate it. The honest way in a notebook, with
no extra dependencies and no moving parts to go stale, is to run the same experiment at
many angles and plot the result as a curve — which is what an animation is anyway, minus
the flicker.

$R_x(\theta)$ rotates the Bloch vector by $\theta$ about the $x$ axis. Starting at the
north pole, that tips $\lvert 0\rangle$ down through the equator to $\lvert 1\rangle$ and
back. We sweep to $4\pi$, two full turns, for a reason that shows up at the end.

In [ ]:
angles = np.linspace(0.0, 4.0 * np.pi, 241)
rx_rows = []
rx_p0 = []
rx_amp0 = []      # the raw amplitude of |0>, phase included

for theta in angles:
    qc = one_qubit(lambda q, t=theta: Rx(q, theta=t))
    rx_rows.append(qc.inspect.bloch_vector(qc.qubits[0]))
    rx_p0.append(qc.inspect.probabilities()[0])
    rx_amp0.append(qc.inspect.amplitude("0"))

rx = np.array(rx_rows)
# np.real of a complex array drops the imaginary part elementwise; here the |0>
# amplitude is real for every Rx angle, so nothing is being thrown away.
rx_amp0 = np.real(np.array(rx_amp0))

fig, ax = plt.subplots(figsize=(9.0, 3.6))
ax.plot(angles, rx[:, 2], lw=2, label="Bloch z")
ax.plot(angles, rx[:, 1], lw=2, label="Bloch y")
ax.plot(angles, rx_p0, lw=2, color="crimson", label="P(0)")
ax.plot(angles, rx_amp0, "k--", lw=1.2, label=r"amplitude of $|0\rangle$")
ax.axvline(2 * np.pi, color="gray", lw=0.8)
ax.set_xticks([0, np.pi, 2 * np.pi, 3 * np.pi, 4 * np.pi], ["0", "π", "2π", "3π", "4π"])
ax.set_xlabel(r"$\theta$")
ax.set_ylim(-1.15, 1.45)
ax.legend(fontsize=8, ncol=4, loc="upper center")
ax.set_title(r"$R_x(\theta)|0\rangle$: everything observable repeats after $2\pi$,"
             " the amplitude does not")
fig.tight_layout()

print("state after Rx(2*pi):", one_qubit(lambda q: Rx(q, theta=2 * np.pi)).inspect.ket())

$z = \cos\theta$, $y = -\sin\theta$, $P(0) = \cos^2(\theta/2)$: the qubit tips over and
comes back, and at the grey line ($\theta = 2\pi$) every observable quantity is exactly
where it started.

The dashed line is not. The amplitude of $\lvert 0\rangle$ is $\cos(\theta/2)$, which at
$\theta = 2\pi$ equals $-1$: $R_x(2\pi) = -I$. A full $360°$ rotation multiplies the
state by $-1$ and needs a *second* full turn to get back to $+1$. That is the half-angle
in every rotation gate's matrix, and it is not a bookkeeping artefact — spin-1/2 particles
really do behave this way, as neutron-interferometry experiments have measured.

Why can we get away with calling that unobservable, then? Because an overall phase on the
*whole* state cancels out of every probability. It becomes observable the moment only
*part* of a superposition is rotated — which is to say, the moment it is a phase
*difference*, which is what the left half of this notebook was about.

## Where to go next

- **[02 — Entanglement](../02-entanglement.ipynb)** and the demo
  [entanglement_and_marginals](entanglement_and_marginals.ipynb): what happens to the
  Bloch vector when a qubit is entangled — it gets *shorter*, and at maximal entanglement
  it vanishes entirely.
- **[05 — Interferometers](../05-interferometers.ipynb)**: the three-gate interferometer
  above, taken seriously as optics.
- **[decoherence_dial](decoherence_dial.ipynb)**: what happens to that $P(0)$ swing when
  something else in the world learns which path the qubit took.

## Assertions

The claims above, re-checked numerically.

In [ ]:
# 1. Z does nothing whatsoever to |0>: same amplitudes, not merely same statistics.
assert np.allclose(one_qubit(Z).inspect.state_vector(), one_qubit().inspect.state_vector())

# 2. Z on |+> gives an orthogonal state with identical z-measurement statistics.
assert np.isclose(abs(overlap), 0.0, atol=1e-12)
assert np.allclose(plus.inspect.probabilities(), minus.inspect.probabilities())
assert np.allclose(plus.inspect.bloch_vector(plus.qubits[0]), (1.0, 0.0, 0.0))
assert np.allclose(minus.inspect.bloch_vector(minus.qubits[0]), (-1.0, 0.0, 0.0))

# 3. While the paths are apart, the phase is invisible to a z-measurement...
assert np.allclose(p0_split, 0.5)
#    ...and the Bloch vector is nonetheless sweeping the whole equator.
assert np.allclose(bloch[:, 0], np.cos(thetas))
assert np.allclose(bloch[:, 1], np.sin(thetas))
assert np.allclose(bloch[:, 2], 0.0, atol=1e-12)

# 4. Recombined, the phase is exactly cos^2(theta/2) of outcome probability.
assert np.allclose(p0_joined, np.cos(thetas / 2) ** 2)

# 5. Rx(2*pi) is -I: every observable is unchanged, the amplitude has flipped sign.
turned = one_qubit(lambda q: Rx(q, theta=2 * np.pi))
assert np.isclose(turned.inspect.amplitude("0").real, -1.0)
assert np.isclose(turned.inspect.fidelity(one_qubit().inspect.state_vector()), 1.0)

# 6. Two kinds of minus: antipodal Bloch vectors are orthogonal states (the most
#    distinguishable pair there is), and the z column carries the P(0) column
#    inside it: P(0) = (1 + z) / 2.
zero, one = one_qubit(), one_qubit(X)
assert np.isclose(abs(zero.inspect.overlap(one.inspect.state_vector())), 0.0, atol=1e-12)
for qc_pair in (zero, one, plus, minus):
    _, _, z_val = qc_pair.inspect.bloch_vector(qc_pair.qubits[0])
    assert np.isclose(qc_pair.inspect.probabilities()[0], (1 + z_val) / 2)

print("all assertions passed")